#### 01 – Fetch GA4 sample data from BigQuery

This notebook downloads the public **GA4 ecommerce sample** dataset from `bigquery-public-data.ga4_obfuscated_sample_ecommerce` and saves one Parquet file per day under `data/raw/`.

**What this notebook does**

- Authenticates to Google Cloud using a service account key
- Loops over the available date range (2020‑11‑01 to 2021‑01‑31)
- Reads each `events_YYYYMMDD` table from BigQuery
- Writes the results to `data/raw/ga4_ecom_YYYYMMDD.parquet`


In [1]:
import os
import datetime
from pathlib import Path

from google.cloud import bigquery
from google.cloud import bigquery_storage
import pyarrow as pa
import pyarrow.parquet as pq

# Project root
# Assumes this notebook lives in ./notebooks and data will be written under ./data/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

# Authentication
# Point GOOGLE_APPLICATION_CREDENTIALS at a service account key JSON
# that has BigQuery read access to the GA4 sample dataset.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./ga4-ecommerce-474507-16c336f9ae68.json"

# Public dataset name under bigquery-public-data
PROJECT = "GA4-ecommerce"
DATASET = "ga4_obfuscated_sample_ecommerce"
TABLE_PREFIX = "events_"  # Prefix for daily tables like events_20201229

# Creates local output folder if it doesn’t exist
OUTPUT_DIR = "./data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Date range used for download (restricted to public sample coverage)
start_date = datetime.date(2020, 11, 1)
end_date = datetime.date(2021, 1, 31)

DATES = []  # List to hold every valid calendar day in the date range
# 'datetime' module handles all calendar complexity (month rollovers, leap years)
# when adding the timedelta, ensuring only valid calendar dates are generated.
current = start_date
while current <= end_date:
    DATES.append(current.strftime("%Y%m%d"))
    current += datetime.timedelta(days=1)
print(f"Number of candidate tables: {len(DATES)}")  # Sanity check: should print 92 days.

# Initializes BigQuery client using Application Default Credentials (ADC).
# Project ID is automatically detected from the environment.
bq_client = bigquery.Client()
bq_storage_client = bigquery_storage.BigQueryReadClient()

# Download loop: one CSV per day under data/raw/
for date in DATES:
    table_id = f"bigquery-public-data.{DATASET}.{TABLE_PREFIX}{date}"
    print(f"Checking {table_id}...")

    try:
        # Load table into a DataFrame via the Storage API
        df = bq_client.list_rows(table_id).to_dataframe(
            bqstorage_client=bq_storage_client,
            progress_bar_type=None,
        )

        # Skip if this date table has no rows
        if df.empty:
            print(f"{date}: Empty table, skipping.")
            continue

        # Save one Parquet file per day in data/raw/
        parquet_path = os.path.join(OUTPUT_DIR, f"ga4_ecom_{date}.parquet")
        # Converts DataFrame df into a PyArrow table
        table = pa.Table.from_pandas(df, preserve_index=False)
        # Writes the PyArrow table to disk
        pq.write_table(table, parquet_path, compression="SNAPPY")

        print(f"Saved {len(df):,} rows → {parquet_path}")

    except Exception as e:
        # Catches missing tables, auth errors, etc
        print(f"Skipping {date} due to error: {e}")

print("Export completed!")

Number of candidate tables: 92
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201101...
Saved 31,272 rows → ./data/raw\ga4_ecom_20201101.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201102...
Saved 48,388 rows → ./data/raw\ga4_ecom_20201102.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201103...
Saved 61,672 rows → ./data/raw\ga4_ecom_20201103.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201104...
Saved 51,866 rows → ./data/raw\ga4_ecom_20201104.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201105...
Saved 51,952 rows → ./data/raw\ga4_ecom_20201105.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201106...
Saved 49,004 rows → ./data/raw\ga4_ecom_20201106.parquet
Checking bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20201107...
Saved 34,309 rows → ./data/raw\ga4_ecom_20201107.parquet
Che